# 🎭 CheapFakeStudio
**Multi-Model Studio** — Generate · Img2Img · Inpaint with model switching



In [ ]:
# @title 🎯 Select Models to Download
# @markdown ### 👇 Select models from here and run this cell first then 1 and 2
# @markdown *(You must select at least one model)*
# @markdown ---
Z_Image_Turbo = False  # @param {type:"boolean"}
# @markdown ⚡ **Z-Image Turbo** — Fast generation only (8 steps, ~6 GB)
FLUX_Klein_4B = False  # @param {type:"boolean"}
# @markdown 🌊 **FLUX.2-klein 4B** — Generation, Img2Img, Inpaint (~7.8 GB + 3.85 GB encoder)
FLUX_Klein_9B = False  # @param {type:"boolean"}
# @markdown 🔮 **FLUX.2-klein 9B Hybrid** — Original 10GB FP8 UNET + GGUF Q3_K_M text encoder
Qwen_Image_Edit = False  # @param {type:"boolean"}


In [ ]:
# @title ⬇️ Step 1: Install & Download Selected Models
if not any([Z_Image_Turbo, FLUX_Klein_4B, FLUX_Klein_9B, Qwen_Image_Edit]):
    raise RuntimeError("❌ No models selected! Go back to the first cell and check at least one model before running this step.")
import os

# ── ComfyUI ───────────────────────────────────────────────
if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
    %cd /content/ComfyUI
    !pip install -q -r requirements.txt

# ComfyUI-GGUF custom nodes (needed for Qwen-Image-Edit AND FLUX.2-klein 9B GGUF)
if (Qwen_Image_Edit or FLUX_Klein_9B) and not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI-GGUF'):
    !git clone https://github.com/city96/ComfyUI-GGUF.git /content/ComfyUI/custom_nodes/ComfyUI-GGUF
    !pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-GGUF/requirements.txt

# Diffusers (needed for Qwen-Image-Edit VAE which ComfyUI can't natively load)
if Qwen_Image_Edit:
    !pip install -q git+https://github.com/huggingface/diffusers

!pip install -q rembg[gpu] onnxruntime-gpu
!pip install -q aria2

# ── Model dirs ────────────────────────────────────────────
DIFF    = '/content/ComfyUI/models/diffusion_models'
CLIP    = '/content/ComfyUI/models/clip'
TXTENC  = '/content/ComfyUI/models/text_encoders'
VAE     = '/content/ComfyUI/models/vae'
for d in [DIFF, CLIP, TXTENC, VAE]:
    os.makedirs(d, exist_ok=True)

def dl(url, dest_dir, name):
    path = os.path.join(dest_dir, name)
    if os.path.exists(path) and os.path.getsize(path) > 1024:
        print(f'  ✅ {name}')
    else:
        if os.path.exists(path):
            os.remove(path)  # remove empty/broken downloads
        print(f'  ⏳ {name}...')
        !aria2c --console-log-level=error -c -x 16 -s 16 -k 1M {url} -d {dest_dir} -o {name}
        print(f'  ✅ {name}')

# ── Z-Image Turbo ─────────────────────────────────────────
if Z_Image_Turbo:
    print('\n📦 Z-Image Turbo FP8')
    dl('https://huggingface.co/T5B/Z-Image-Turbo-FP8/resolve/main/z-image-turbo-fp8-e4m3fn.safetensors', DIFF, 'z-image-turbo-fp8-e4m3fn.safetensors')
    dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/text_encoders/qwen_3_4b.safetensors', CLIP, 'qwen_3_4b.safetensors')
    dl('https://huggingface.co/Comfy-Org/z_image_turbo/resolve/main/split_files/vae/ae.safetensors', VAE, 'ae.safetensors')

# ── FLUX.2 shared VAE (needed by both 9B and 4B) ──────────
if FLUX_Klein_9B or FLUX_Klein_4B:
    dl('https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-9b/resolve/main/split_files/vae/flux2-vae.safetensors', VAE, 'flux2-vae.safetensors')

# ── FLUX.2-klein 4B ───────────────────────────────────────
if FLUX_Klein_4B:
    print('\n📦 FLUX.2-klein 4B')
    dl('https://huggingface.co/black-forest-labs/FLUX.2-klein-4B/resolve/main/flux-2-klein-4b.safetensors', DIFF, 'flux-2-klein-4b.safetensors')
    dl('https://huggingface.co/Comfy-Org/vae-text-encorder-for-flux-klein-4b/resolve/main/split_files/text_encoders/qwen_3_4b_fp4_flux2.safetensors', TXTENC, 'qwen_3_4b_fp4_flux2.safetensors')

# ── FLUX.2-klein 9B (Safetensors UNET + GGUF CLIP) ────────
if FLUX_Klein_9B:
    print('\n📦 FLUX.2-klein 9B (10GB UNET + 3.6GB GGUF CLIP Q3_K_M)')
    dl('https://huggingface.co/black-forest-labs/FLUX.2-klein-9b-kv-fp8/resolve/main/flux-2-klein-9b-kv-fp8.safetensors', DIFF, 'flux-2-klein-9b-kv-fp8.safetensors')
    dl('https://huggingface.co/unsloth/Qwen3-8B-GGUF/resolve/main/Qwen3-8B-Q3_K_M.gguf', TXTENC, 'Qwen3-8B-Q3_K_M.gguf')

# ── Qwen-Image-Edit-2511 GGUF Q4_0 ────────────────────────
if Qwen_Image_Edit:
    print('\n📦 Qwen-Image-Edit-2511 (11.9GB Q4_0 UNET + 3GB Q2_K Text Encoder + mmproj)')
    dl('https://huggingface.co/unsloth/Qwen-Image-Edit-2511-GGUF/resolve/main/qwen-image-edit-2511-Q4_0.gguf', DIFF, 'qwen-image-edit-2511-Q4_0.gguf')
    dl('https://huggingface.co/unsloth/Qwen2.5-VL-7B-Instruct-GGUF/resolve/main/Qwen2.5-VL-7B-Instruct-Q2_K.gguf', CLIP, 'Qwen2.5-VL-7B-Instruct-Q2_K.gguf')
    # mmproj: file is 'mmproj-F16.gguf' on HF but ComfyUI-GGUF matches by model prefix,
    # so we save it as 'Qwen2.5-VL-7B-Instruct-mmproj-F16.gguf' for auto-detection
    dl('https://huggingface.co/unsloth/Qwen2.5-VL-7B-Instruct-GGUF/resolve/main/mmproj-F16.gguf', CLIP, 'Qwen2.5-VL-7B-Instruct-mmproj-F16.gguf')
    dl('https://huggingface.co/Qwen/Qwen-Image-Edit-2511/resolve/main/vae/diffusion_pytorch_model.safetensors', VAE, 'qwen_image_vae.safetensors')

# Symlink clip/ ↔ text_encoders/ (Some nodes search clip, others text_encoders)
import glob
for f in glob.glob(f'{CLIP}/*'):
    link = os.path.join(TXTENC, os.path.basename(f))
    if not os.path.exists(link):
        os.symlink(f, link)
for f in glob.glob(f'{TXTENC}/*'):
    link = os.path.join(CLIP, os.path.basename(f))
    if not os.path.exists(link):
        os.symlink(f, link)



In [ ]:
# @title 🚀 Step 2: Launch CheapFakeStudio
import os
os.chdir('/content')
REPO = 'https://github.com/MuntasirMalek/CheapFakeStudio.git'
if os.path.exists('/content/CheapFakeStudio'):
    %cd /content/CheapFakeStudio
    !git pull --ff-only || true
else:
    !git clone {REPO}
    %cd /content/CheapFakeStudio
!python app.py
